# black-box regression results: point plots

Generates the point plot figures comparing performance on the black-box regression problems.

In [ ]:
import pandas as pd
import json
import os
import numpy as np; np.random.seed(42)
from glob import glob
from tqdm import tqdm
import seaborn as sns
import pdb

import sys; sys.path.append('..')

import matplotlib.pyplot as plt
import matplotlib

matplotlib.rc('pdf', fonttype=42)
matplotlib.rc('ps', fonttype=42)

params = {'mathtext.default': 'regular' }          
plt.rcParams.update(params)

sns.set(font_scale=1.0, style='whitegrid')
sns.set(palette='magma')
sns.set_style("whitegrid")

plt.rcParams.update({
    'font.family': 'serif',
    'font.serif': ['Nimbus Roman', 'Times New Roman', 'Liberation Serif', 'DejaVu Serif'],
    'font.size': 11,
    'axes.titlesize': 11,
    'axes.labelsize': 11,
    'xtick.labelsize': 11,
    'ytick.labelsize': 11,
    'legend.fontsize': 11,
    'legend.title_fontsize': 11,
})

def is_rasr(name):
    return 'RASR' in name

NSR_METHODS = {'E2E', 'SNIP', 'NeSymRes', 'TPSR', 'Flash-ANSR'}  # uDSR excluded: retrained per dataset, so not amortized
def is_nsr(name):
    return name in NSR_METHODS or is_rasr(name)
NSR_MARKER = '■'
NSR_MARKER_COLOR = '#2a78d6'

if True: # Off-the-shelf evaluation
    rdir  = '../../results/black-box/'
    figdir = f'../figs/black-box/'
else: # with HP tuning
    rdir  = '../../results/black-box-tuned/'
    figdir = f'../figs/black-box-tuned/'

datadir = '../../datasets/blackbox/'

pretty_labels = {'model_size': 'Mean Expression Size', 'r2_test_rank': 'Mean Test $R^2$ Rank', 'r2_test_norm': 'Mean Normalized $R^2$'}

def save(name='tmp',h=None):
    name = name.strip().replace(' ','-').replace('%','pct')
    if h == None:
        h = plt.gcf()

    h.tight_layout()

    print('saving',figdir+'/'+name+'.pdf')

    if not os.path.exists(figdir):
        os.makedirs(figdir)

    plt.savefig(figdir+'/'+name+'.pdf', bbox_inches='tight')


# read data from feather

In [ ]:
df_results = pd.read_feather(rdir+'results.feather')
df_sum = pd.read_csv(rdir+'results-summary.csv.gz', compression='gzip')

# -1 marks a failed complexity extraction; mask it to NaN so means/medians skip it
df_results['model_size'] = df_results['model_size'].replace(-1, np.nan)
df_sum['model_size'] = df_sum['model_size'].replace(-1, np.nan)

df_results = df_results[~df_results['algorithm'].isin(['Genetic Engine rs', 'Genetic Engine hc', 'Genetic Engine 1p1'])]
df_sum = df_sum[~df_sum['algorithm'].isin(['Genetic Engine rs', 'Genetic Engine hc', 'Genetic Engine 1p1'])]

df_results = df_results.replace({'Genetic Engine': 'G. Engine'})
df_sum = df_sum.replace({'Genetic Engine': 'G. Engine'})

print('mean trial count:')
print(df_results.groupby('algorithm')['dataset'].count().sort_values()
      / df_results.dataset.nunique())

In [ ]:
from scripts.impute_missing_runs import impute_missing_runs

_impute_missing_runs = True
_impute_r2_source     = 'fixed'  # 'fixed' or 'linear'
_impute_r2_test       = 0.0
_impute_model_size    = "dataset_median"  # 'dataset_median', 'algorithm_median', 'algorithm_mean', 'linear_expression_size', or a fixed number
_impute_algorithms    = None
_linear_algorithm_name = 'Linear'

if _impute_missing_runs:
    df_results, df_sum = impute_missing_runs(
        df_results, df_sum,
        datadir=datadir,
        impute_r2_source=_impute_r2_source,
        impute_r2_test=_impute_r2_test,
        impute_model_size=_impute_model_size,
        impute_algorithms=_impute_algorithms,
        linear_algorithm_name=_linear_algorithm_name,
    )


In [ ]:
_r2_norm_strategy = 'median'  # 'minmax': (method - worst) / (best - worst)
                               # 'median': (method - median) / (best - median)
_r2_norm_clip_negative = True  # clip normalized scores below 0 to 0 (keeps the scale in [0, 1] with the median strategy)
_R2_NORM_MIN_DENOM = 1e-5
_IMPROVABILITY_MIN_ERR = 1e-12

def _normalize_r2(x, strategy, clip_negative=None):
    clip_negative = _r2_norm_clip_negative if clip_negative is None else clip_negative
    best = x.max()
    baseline = x.min() if strategy == 'minmax' else x.median()
    denom = best - baseline
    if denom == 0:
        return pd.Series(1.0, index=x.index)
    norm = (x - baseline) / max(denom, _R2_NORM_MIN_DENOM)
    return norm.clip(lower=0) if clip_negative else norm

def _improvability(err):
    """Relative headroom to the best method in percent, (err - best_err) / err * 100 (lower is better)."""
    best = err.min()
    return ((err - best) / err.clip(lower=_IMPROVABILITY_MIN_ERR) * 100).clip(lower=0, upper=100)

def _apply_excluding_similar(group, col, fn, decimals=3, cohort=None):
    """Apply `fn` to `col` within a dataset, with reference stats from the non-rasr methods plus each rasr cohort."""
    is_special = group['*algorithm*'].str.contains('rasr', case=False)
    out = pd.Series(index=group.index, dtype=float)
    base = group.loc[~is_special, col]
    out.loc[~is_special] = fn(base)

    special_idx = group.index[is_special]
    labels = pd.Series(special_idx, index=special_idx, dtype=object)
    if cohort is not None:
        given = cohort.reindex(special_idx)
        labels.loc[given.notna()] = given.loc[given.notna()]

    for _, idxs in labels.groupby(labels, sort=False).groups.items():
        idxs = list(idxs)
        combined = pd.concat([base, group.loc[idxs, col]])
        scored = fn(combined)
        for idx in idxs:
            out.loc[idx] = round(scored.loc[idx], decimals)
    return out

def r2_norm_excluding_similar(group, r2_col='r2_test', strategy=None, clip_negative=None, cohort=None):
    strategy = strategy or _r2_norm_strategy
    return _apply_excluding_similar(
        group, r2_col, lambda x: _normalize_r2(x, strategy, clip_negative), cohort=cohort)

def improvability_excluding_similar(group, err_col='rmse_test', cohort=None):
    return _apply_excluding_similar(group, err_col, _improvability, cohort=cohort)


In [ ]:
def compare(df_compare, x='r2_test',y='algorithm', row=None, col=None, scale=None, xlim=[],**kwargs):
    plt.figure()
    order = df_compare.groupby(y)[x].median().sort_values(ascending=False).index
    if scale=='log' and len(xlim)>0 and xlim[0] == 0:
        df_compare.loc[:,x] += 1
        xlim[0] = 1
        xnew = '1 + '+x
        df_compare=df_compare.rename(columns={x:xnew})
        x = xnew
    
    sns.catplot(data=df_compare, 
                kind='point',
                y=y,
                x=x,
                order=order,
                fliersize=0,
                row=row,
                col=col,
                **kwargs
               )

    plt.ylabel('')
    if len(xlim)>0:
        plt.xlim(xlim[0],xlim[1])
    if scale:
        plt.gca().set_xscale(scale)
    
    save(name='_'.join(['cat-boxplot',x + '-by-'+ y]) )
    if col:
        save(name='_'.join(['cat-boxplot',x + '-by-'+ y] + [col]) )



# excluded_datasets = ['1089_USCrime', '1193_BNG_lowbwt', '1199_BNG_echoMonths', '650_fri_c0_500_50']
# excluded_datasets = ['1089_USCrime', '1193_BNG_lowbwt', '1199_BNG_echoMonths', '650_fri_c0_500_50', '678_visualizing_environmental', '210_cloud', '192_vineyard']
excluded_datasets = []

excluded_methods = ['Linear']  # 'Linear' is only an imputation source; dropped from all plots and normalizations

alg_name_map = df_sum[['algorithm', '*algorithm*']].drop_duplicates().set_index('algorithm')['*algorithm*']

df_sum_filtered = df_sum[~df_sum['dataset'].isin(excluded_datasets)].copy()

df_sum_filtered['*algorithm*'] = df_sum_filtered['*algorithm*'].str.lstrip('*')


df_sum_filtered = df_sum_filtered[~df_sum_filtered['*algorithm*'].isin(excluded_methods)]

print(f"Remaining: {df_sum_filtered['dataset'].nunique()} datasets")
print(df_sum_filtered['dataset'].unique())

## point plot comparisons on PMLB (mean over datasets)

In [ ]:
# --- configure here ---
_rasr_method = "rasr_94m"  # single rasr method to show (others dropped); None keeps all
_show_extra_variants = False  # also show the variants in _extra_variant_rename
_seed_agg   = 'median'  # 'mean' or 'median' over the per-seed trials of each dataset+algorithm
_score_threshold = -1  # methods whose mean x_vars[0] is below this are dropped; None disables.
# NOTE: for lower-is-better metrics (e.g. r2_test_rank) this drops the best methods; invert the
# sign or use r2_test_norm.

x_vars = [
#       'rmse_test',
#         'log_mse_test',
#         'r2_test',
        'r2_test_rank',
#         'rmse_test_improvability',
#         'rmse_test_improvability',
         'model_size',
#         'model_size_rank',
        'training time (s)',
]
# ----------------------

_extra_variant_rename = {
    'rasr_94m_lambda0.15': 'RASR-94M-M',
    'rasr_94m_lambda0.5':  'RASR-94M-S',
    'rasr_24m':            'RASR-24M-L',
    'rasr_94m_base_c1024': 'RASR-94M-base',
}

# start from the per-seed df_results (df_sum is already collapsed over seeds)

df_plot = df_results.copy()
df_plot = df_plot[~df_plot['dataset'].isin(excluded_datasets)]
df_plot['*algorithm*'] = df_plot['algorithm'].map(alg_name_map).str.lstrip('*')
df_plot = df_plot[~df_plot['*algorithm*'].isin(excluded_methods)]

if _rasr_method is not None:
    _keep_names = {_rasr_method}
    if _show_extra_variants:
        _keep_names |= set(_extra_variant_rename)
    is_rasr_mask = df_plot['*algorithm*'].str.contains('rasr', case=False)
    df_plot = df_plot[~is_rasr_mask | df_plot['*algorithm*'].isin(_keep_names)]

x_vars_raw = ['r2_test', 'mse_test', 'model_size', 'training time (s)']

df_plot = df_plot.groupby(['*algorithm*', 'dataset'], as_index=False)[x_vars_raw].agg(_seed_agg)

def _rank_excluding_similar(group):
    is_special = group['*algorithm*'].str.contains('rasr', case=False)
    ranks = pd.Series(index=group.index, dtype=float)
    base = group.loc[~is_special, 'r2_test']
    ranks.loc[~is_special] = base.rank(ascending=False)
    for idx in group.index[is_special]:
        combined = pd.concat([base, group.loc[[idx], 'r2_test']])
        ranks.loc[idx] = round(combined.rank(ascending=False).loc[idx], 3)
    return ranks

df_plot['r2_test_rank'] = df_plot.groupby('dataset', group_keys=False).apply(_rank_excluding_similar)

df_plot['r2_test_norm'] = df_plot.groupby('dataset', group_keys=False).apply(r2_norm_excluding_similar)

df_plot['rmse_test'] = np.sqrt(df_plot['mse_test'])
df_plot['rmse_test_improvability'] = df_plot.groupby('dataset', group_keys=False).apply(improvability_excluding_similar)

# display-only rename of the shown rasr methods (after the name-based logic above)
if _rasr_method is not None:
    _rename_map = {_rasr_method: 'RASR-94M-L'}
    if _show_extra_variants:
        _rename_map.update(_extra_variant_rename)
    df_plot['*algorithm*'] = df_plot['*algorithm*'].replace(_rename_map)

df_plot['*algorithm*'] = df_plot['*algorithm*'].replace({'PYSR': 'PySR'})

if _score_threshold is not None:
    _method_scores = df_plot.groupby('*algorithm*')[x_vars[0]].mean()
    _keep_methods = _method_scores[_method_scores >= _score_threshold].index
    df_plot = df_plot[df_plot['*algorithm*'].isin(_keep_methods)]

order = df_plot.groupby('*algorithm*')[x_vars[0]].mean().sort_values(
                    ascending='r2' not in x_vars[0] or 'rank' in x_vars[0] or 'improvability' in x_vars[0]).index

with plt.rc_context({
    'font.size': 7, 'axes.titlesize': 7, 'axes.labelsize': 7,
    'xtick.labelsize': 7, 'ytick.labelsize': 7,
    'legend.fontsize': 7, 'legend.title_fontsize': 7,
}):
    g = sns.PairGrid(df_plot,
                     x_vars=x_vars,
                     y_vars=['*algorithm*'],
                     height=3.8,
                     aspect=0.4843,
                     hue_order=order,
                     palette=sns.color_palette("rocket", n_colors=len(order)),
                    )

    g.map(sns.pointplot,
          orient="h",
          order=order,
          palette=sns.color_palette("rocket", n_colors=len(order)),
          join=False,
          estimator=np.mean,
          n_boot=1000,
          ci=95,
          markersize=4.3,
          linewidth=1.6,
          err_kws={'linewidth': 1.6},
         )

    titles = [pretty_labels.get(x, x.replace('_',' ').title().replace('(S)','(s)').replace('R2','$R^2$')) for x in x_vars]
    g.axes.flat[0].set_ylabel('')

    for ax, title in zip(g.axes.flat, titles):
        ax.set(title=title)
        ax.set_xlabel('')

        if any([n in title.lower() for n in ['size','time']]):
            ax.set_xscale('log')

        if title == '$R^2$ Test':
            ax.set_xlim([-.25,1])

        ax.yaxis.grid(True)

    # bold RASR entries; small square after neural methods
    g.fig.tight_layout(w_pad=0.4)
    g.fig.canvas.draw()
    _renderer = g.fig.canvas.get_renderer()
    _ax0 = g.axes.flat[0]
    for _label in _ax0.get_yticklabels():
        _name = _label.get_text()
        if is_rasr(_name):
            _label.set_fontweight('bold')
        if is_nsr(_name):
            _bbox = _label.get_window_extent(_renderer)
            _fx, _fy = g.fig.transFigure.inverted().transform(
                (_bbox.x1 + 2, (_bbox.y0 + _bbox.y1) / 2))
            g.fig.text(_fx, _fy, NSR_MARKER, color=NSR_MARKER_COLOR, fontsize=7,
                        va='center', ha='left', transform=g.fig.transFigure)

    sns.despine(left=True, bottom=True)

    # inlined save(): its tight_layout() would undo w_pad=0.4
    _name = "r2_test_rank_overview_plot"
    print('saving', figdir+'/'+_name+'.pdf')
    if not os.path.exists(figdir):
        os.makedirs(figdir)
    plt.savefig(figdir+'/'+_name+'.pdf', bbox_inches='tight')

## point plot comparisons — appendix variants

As above, but also showing RASR-24M-L, RASR-94M-M, RASR-94M-S and RASR-94M-base alongside RASR-94M-L: once over all twelve datasets, and once restricted to the eight datasets RASR neither subsamples nor feature-reduces.

In [ ]:
# --- configure here ---
_rasr_method = "rasr_94m"  # single rasr method to show (others dropped); None keeps all
_show_extra_variants = True  # appendix variant: also show the variants in _extra_variant_rename
_seed_agg   = 'median'  # 'mean' or 'median' over the per-seed trials of each dataset+algorithm
_score_threshold = -1  # methods whose mean x_vars[0] is below this are dropped; None disables.
# NOTE: for lower-is-better metrics (e.g. r2_test_rank) this drops the best methods; invert the
# sign or use r2_test_norm.

x_vars = [
#       'rmse_test',
#         'log_mse_test',
#         'r2_test',
        'r2_test_rank',
#         'rmse_test_improvability',
#         'rmse_test_improvability',
         'model_size',
#         'model_size_rank',
        'training time (s)',
]
# ----------------------

_extra_variant_rename = {
    'rasr_94m_lambda0.15': 'RASR-94M-M',
    'rasr_94m_lambda0.5':  'RASR-94M-S',
    'rasr_24m':            'RASR-24M-L',
    'rasr_94m_base_c1024': 'RASR-94M-base',
}

# start from the per-seed df_results (df_sum is already collapsed over seeds)

df_plot = df_results.copy()
df_plot = df_plot[~df_plot['dataset'].isin(excluded_datasets)]
df_plot['*algorithm*'] = df_plot['algorithm'].map(alg_name_map).str.lstrip('*')
df_plot = df_plot[~df_plot['*algorithm*'].isin(excluded_methods)]

if _rasr_method is not None:
    _keep_names = {_rasr_method}
    if _show_extra_variants:
        _keep_names |= set(_extra_variant_rename)
    is_rasr_mask = df_plot['*algorithm*'].str.contains('rasr', case=False)
    df_plot = df_plot[~is_rasr_mask | df_plot['*algorithm*'].isin(_keep_names)]

x_vars_raw = ['r2_test', 'mse_test', 'model_size', 'training time (s)']

df_plot = df_plot.groupby(['*algorithm*', 'dataset'], as_index=False)[x_vars_raw].agg(_seed_agg)

def _rank_excluding_similar(group):
    is_special = group['*algorithm*'].str.contains('rasr', case=False)
    ranks = pd.Series(index=group.index, dtype=float)
    base = group.loc[~is_special, 'r2_test']
    ranks.loc[~is_special] = base.rank(ascending=False)
    for idx in group.index[is_special]:
        combined = pd.concat([base, group.loc[[idx], 'r2_test']])
        ranks.loc[idx] = round(combined.rank(ascending=False).loc[idx], 3)
    return ranks

df_plot['r2_test_rank'] = df_plot.groupby('dataset', group_keys=False).apply(_rank_excluding_similar)

df_plot['r2_test_norm'] = df_plot.groupby('dataset', group_keys=False).apply(r2_norm_excluding_similar)

df_plot['rmse_test'] = np.sqrt(df_plot['mse_test'])
df_plot['rmse_test_improvability'] = df_plot.groupby('dataset', group_keys=False).apply(improvability_excluding_similar)

# display-only rename of the shown rasr methods (after the name-based logic above)
if _rasr_method is not None:
    _rename_map = {_rasr_method: 'RASR-94M-L'}
    if _show_extra_variants:
        _rename_map.update(_extra_variant_rename)
    df_plot['*algorithm*'] = df_plot['*algorithm*'].replace(_rename_map)

df_plot['*algorithm*'] = df_plot['*algorithm*'].replace({'PYSR': 'PySR'})

if _score_threshold is not None:
    _method_scores = df_plot.groupby('*algorithm*')[x_vars[0]].mean()
    _keep_methods = _method_scores[_method_scores >= _score_threshold].index
    df_plot = df_plot[df_plot['*algorithm*'].isin(_keep_methods)]

order = df_plot.groupby('*algorithm*')[x_vars[0]].mean().sort_values(
                    ascending='r2' not in x_vars[0] or 'rank' in x_vars[0] or 'improvability' in x_vars[0]).index

with plt.rc_context({
    'font.size': 7, 'axes.titlesize': 7, 'axes.labelsize': 7,
    'xtick.labelsize': 7, 'ytick.labelsize': 7,
    'legend.fontsize': 7, 'legend.title_fontsize': 7,
}):
    g = sns.PairGrid(df_plot,
                     x_vars=x_vars,
                     y_vars=['*algorithm*'],
                     height=3.8,
                     aspect=0.4843,
                     hue_order=order,
                     palette=sns.color_palette("rocket", n_colors=len(order)),
                    )

    g.map(sns.pointplot,
          orient="h",
          order=order,
          palette=sns.color_palette("rocket", n_colors=len(order)),
          join=False,
          estimator=np.mean,
          n_boot=1000,
          ci=95,
          markersize=4.3,
          linewidth=1.6,
          err_kws={'linewidth': 1.6},
         )

    titles = [pretty_labels.get(x, x.replace('_',' ').title().replace('(S)','(s)').replace('R2','$R^2$')) for x in x_vars]
    g.axes.flat[0].set_ylabel('')

    for ax, title in zip(g.axes.flat, titles):
        ax.set(title=title)
        ax.set_xlabel('')

        if any([n in title.lower() for n in ['size','time']]):
            ax.set_xscale('log')

        if title == '$R^2$ Test':
            ax.set_xlim([-.25,1])

        ax.yaxis.grid(True)

    # bold RASR entries; small square after neural methods
    g.fig.tight_layout(w_pad=0.4)
    g.fig.canvas.draw()
    _renderer = g.fig.canvas.get_renderer()
    _ax0 = g.axes.flat[0]
    for _label in _ax0.get_yticklabels():
        _name = _label.get_text()
        if is_rasr(_name):
            _label.set_fontweight('bold')
        if is_nsr(_name):
            _bbox = _label.get_window_extent(_renderer)
            _fx, _fy = g.fig.transFigure.inverted().transform(
                (_bbox.x1 + 2, (_bbox.y0 + _bbox.y1) / 2))
            g.fig.text(_fx, _fy, NSR_MARKER, color=NSR_MARKER_COLOR, fontsize=7,
                        va='center', ha='left', transform=g.fig.transFigure)

    sns.despine(left=True, bottom=True)

    # inlined save(): its tight_layout() would undo w_pad=0.4
    _name = "r2_test_rank_overview_plot_full"
    print('saving', figdir+'/'+_name+'.pdf')
    if not os.path.exists(figdir):
        os.makedirs(figdir)
    plt.savefig(figdir+'/'+_name+'.pdf', bbox_inches='tight')

In [ ]:
# --- configure here ---
_rasr_method = "rasr_94m"  # single rasr method to show (others dropped); None keeps all
_show_extra_variants = True  # appendix variant: also show the variants in _extra_variant_rename
_seed_agg   = 'median'  # 'mean' or 'median' over the per-seed trials of each dataset+algorithm
_score_threshold = -1  # methods whose mean x_vars[0] is below this are dropped; None disables.
# NOTE: for lower-is-better metrics (e.g. r2_test_rank) this drops the best methods; invert the
# sign or use r2_test_norm.

x_vars = [
#       'rmse_test',
#         'log_mse_test',
#         'r2_test',
        'r2_test_rank',
#         'rmse_test_improvability',
#         'rmse_test_improvability',
         'model_size',
#         'model_size_rank',
        'training time (s)',
]
# ----------------------

_extra_variant_rename = {
    'rasr_94m_lambda0.15': 'RASR-94M-M',
    'rasr_94m_lambda0.5':  'RASR-94M-S',
    'rasr_24m':            'RASR-24M-L',
    'rasr_94m_base_c1024': 'RASR-94M-base',
}

# start from the per-seed df_results (df_sum is already collapsed over seeds)

df_plot = df_results.copy()
# appendix variant: also drop the four datasets RASR subsamples or feature-reduces
_oversized_datasets = ['1089_USCrime', '1193_BNG_lowbwt', '1199_BNG_echoMonths', '650_fri_c0_500_50']
df_plot = df_plot[~df_plot['dataset'].isin(excluded_datasets) & ~df_plot['dataset'].isin(_oversized_datasets)]
df_plot['*algorithm*'] = df_plot['algorithm'].map(alg_name_map).str.lstrip('*')
df_plot = df_plot[~df_plot['*algorithm*'].isin(excluded_methods)]

if _rasr_method is not None:
    _keep_names = {_rasr_method}
    if _show_extra_variants:
        _keep_names |= set(_extra_variant_rename)
    is_rasr_mask = df_plot['*algorithm*'].str.contains('rasr', case=False)
    df_plot = df_plot[~is_rasr_mask | df_plot['*algorithm*'].isin(_keep_names)]

x_vars_raw = ['r2_test', 'mse_test', 'model_size', 'training time (s)']

df_plot = df_plot.groupby(['*algorithm*', 'dataset'], as_index=False)[x_vars_raw].agg(_seed_agg)

def _rank_excluding_similar(group):
    is_special = group['*algorithm*'].str.contains('rasr', case=False)
    ranks = pd.Series(index=group.index, dtype=float)
    base = group.loc[~is_special, 'r2_test']
    ranks.loc[~is_special] = base.rank(ascending=False)
    for idx in group.index[is_special]:
        combined = pd.concat([base, group.loc[[idx], 'r2_test']])
        ranks.loc[idx] = round(combined.rank(ascending=False).loc[idx], 3)
    return ranks

df_plot['r2_test_rank'] = df_plot.groupby('dataset', group_keys=False).apply(_rank_excluding_similar)

df_plot['r2_test_norm'] = df_plot.groupby('dataset', group_keys=False).apply(r2_norm_excluding_similar)

df_plot['rmse_test'] = np.sqrt(df_plot['mse_test'])
df_plot['rmse_test_improvability'] = df_plot.groupby('dataset', group_keys=False).apply(improvability_excluding_similar)

# display-only rename of the shown rasr methods (after the name-based logic above)
if _rasr_method is not None:
    _rename_map = {_rasr_method: 'RASR-94M-L'}
    if _show_extra_variants:
        _rename_map.update(_extra_variant_rename)
    df_plot['*algorithm*'] = df_plot['*algorithm*'].replace(_rename_map)

df_plot['*algorithm*'] = df_plot['*algorithm*'].replace({'PYSR': 'PySR'})

if _score_threshold is not None:
    _method_scores = df_plot.groupby('*algorithm*')[x_vars[0]].mean()
    _keep_methods = _method_scores[_method_scores >= _score_threshold].index
    df_plot = df_plot[df_plot['*algorithm*'].isin(_keep_methods)]

order = df_plot.groupby('*algorithm*')[x_vars[0]].mean().sort_values(
                    ascending='r2' not in x_vars[0] or 'rank' in x_vars[0] or 'improvability' in x_vars[0]).index

with plt.rc_context({
    'font.size': 7, 'axes.titlesize': 7, 'axes.labelsize': 7,
    'xtick.labelsize': 7, 'ytick.labelsize': 7,
    'legend.fontsize': 7, 'legend.title_fontsize': 7,
}):
    g = sns.PairGrid(df_plot,
                     x_vars=x_vars,
                     y_vars=['*algorithm*'],
                     height=3.8,
                     aspect=0.4843,
                     hue_order=order,
                     palette=sns.color_palette("rocket", n_colors=len(order)),
                    )

    g.map(sns.pointplot,
          orient="h",
          order=order,
          palette=sns.color_palette("rocket", n_colors=len(order)),
          join=False,
          estimator=np.mean,
          n_boot=1000,
          ci=95,
          markersize=4.3,
          linewidth=1.6,
          err_kws={'linewidth': 1.6},
         )

    titles = [pretty_labels.get(x, x.replace('_',' ').title().replace('(S)','(s)').replace('R2','$R^2$')) for x in x_vars]
    g.axes.flat[0].set_ylabel('')

    for ax, title in zip(g.axes.flat, titles):
        ax.set(title=title)
        ax.set_xlabel('')

        if any([n in title.lower() for n in ['size','time']]):
            ax.set_xscale('log')

        if title == '$R^2$ Test':
            ax.set_xlim([-.25,1])

        ax.yaxis.grid(True)

    # bold RASR entries; small square after neural methods
    g.fig.tight_layout(w_pad=0.4)
    g.fig.canvas.draw()
    _renderer = g.fig.canvas.get_renderer()
    _ax0 = g.axes.flat[0]
    for _label in _ax0.get_yticklabels():
        _name = _label.get_text()
        if is_rasr(_name):
            _label.set_fontweight('bold')
        if is_nsr(_name):
            _bbox = _label.get_window_extent(_renderer)
            _fx, _fy = g.fig.transFigure.inverted().transform(
                (_bbox.x1 + 2, (_bbox.y0 + _bbox.y1) / 2))
            g.fig.text(_fx, _fy, NSR_MARKER, color=NSR_MARKER_COLOR, fontsize=7,
                        va='center', ha='left', transform=g.fig.transFigure)

    sns.despine(left=True, bottom=True)

    # inlined save(): its tight_layout() would undo w_pad=0.4
    _name = "r2_test_rank_overview_plot_small_datasets"
    print('saving', figdir+'/'+_name+'.pdf')
    if not os.path.exists(figdir):
        os.makedirs(figdir)
    plt.savefig(figdir+'/'+_name+'.pdf', bbox_inches='tight')

## point plot comparisons — selected methods

As the main point plot, but hiding BSR, FFX, the AFP methods, GPZGD, ITEA, EPLEX and FEAT. Ranks are still computed against all baselines.

In [ ]:
# --- configure here ---
_rasr_method = "rasr_94m"  # single rasr method to show (others dropped); None keeps all
_show_extra_variants = False  # also show the variants in _extra_variant_rename
_seed_agg   = 'median'  # 'mean' or 'median' over the per-seed trials of each dataset+algorithm
_score_threshold = -1  # methods whose mean x_vars[0] is below this are dropped; None disables.
# NOTE: for lower-is-better metrics (e.g. r2_test_rank) this drops the best methods; invert the
# sign or use r2_test_norm.

x_vars = [
#       'rmse_test',
#         'log_mse_test',
#         'r2_test',
        'r2_test_rank',
#         'rmse_test_improvability',
#         'rmse_test_improvability',
         'model_size',
#         'model_size_rank',
        'training time (s)',
]
# ----------------------

_extra_variant_rename = {
    'rasr_94m_lambda0.15': 'RASR-94M-M',
    'rasr_94m_lambda0.5':  'RASR-94M-S',
    'rasr_24m':            'RASR-24M-L',
    'rasr_94m_base_c1024': 'RASR-94M-base',
}

# start from the per-seed df_results (df_sum is already collapsed over seeds)

df_plot = df_results.copy()
df_plot = df_plot[~df_plot['dataset'].isin(excluded_datasets)]
df_plot['*algorithm*'] = df_plot['algorithm'].map(alg_name_map).str.lstrip('*')
df_plot = df_plot[~df_plot['*algorithm*'].isin(excluded_methods)]

if _rasr_method is not None:
    _keep_names = {_rasr_method}
    if _show_extra_variants:
        _keep_names |= set(_extra_variant_rename)
    is_rasr_mask = df_plot['*algorithm*'].str.contains('rasr', case=False)
    df_plot = df_plot[~is_rasr_mask | df_plot['*algorithm*'].isin(_keep_names)]

x_vars_raw = ['r2_test', 'mse_test', 'model_size', 'training time (s)']

df_plot = df_plot.groupby(['*algorithm*', 'dataset'], as_index=False)[x_vars_raw].agg(_seed_agg)

def _rank_excluding_similar(group):
    is_special = group['*algorithm*'].str.contains('rasr', case=False)
    ranks = pd.Series(index=group.index, dtype=float)
    base = group.loc[~is_special, 'r2_test']
    ranks.loc[~is_special] = base.rank(ascending=False)
    for idx in group.index[is_special]:
        combined = pd.concat([base, group.loc[[idx], 'r2_test']])
        ranks.loc[idx] = round(combined.rank(ascending=False).loc[idx], 3)
    return ranks

df_plot['r2_test_rank'] = df_plot.groupby('dataset', group_keys=False).apply(_rank_excluding_similar)

df_plot['r2_test_norm'] = df_plot.groupby('dataset', group_keys=False).apply(r2_norm_excluding_similar)

df_plot['rmse_test'] = np.sqrt(df_plot['mse_test'])
df_plot['rmse_test_improvability'] = df_plot.groupby('dataset', group_keys=False).apply(improvability_excluding_similar)

# display-only rename of the shown rasr methods (after the name-based logic above)
if _rasr_method is not None:
    _rename_map = {_rasr_method: 'RASR-94M-L'}
    if _show_extra_variants:
        _rename_map.update(_extra_variant_rename)
    df_plot['*algorithm*'] = df_plot['*algorithm*'].replace(_rename_map)

df_plot['*algorithm*'] = df_plot['*algorithm*'].replace({'PYSR': 'PySR'})

if _score_threshold is not None:
    _method_scores = df_plot.groupby('*algorithm*')[x_vars[0]].mean()
    _keep_methods = _method_scores[_method_scores >= _score_threshold].index
    df_plot = df_plot[df_plot['*algorithm*'].isin(_keep_methods)]

# hide methods from the plot only, after ranking/normalization
_hidden_methods = ['BSR', 'FFX', 'AFP', 'AFP_FE', 'AFP_EHC', 'GPZGD', 'ITEA', 'EPLEX', 'FEAT']
df_plot = df_plot[~df_plot['*algorithm*'].isin(_hidden_methods)]

order = df_plot.groupby('*algorithm*')[x_vars[0]].mean().sort_values(
                    ascending='r2' not in x_vars[0] or 'rank' in x_vars[0] or 'improvability' in x_vars[0]).index

_height = 3.8 * len(order) / 28
_panel_width = 3.8 * 0.4843

with plt.rc_context({
    'font.size': 7, 'axes.titlesize': 7, 'axes.labelsize': 7,
    'xtick.labelsize': 7, 'ytick.labelsize': 7,
    'legend.fontsize': 7, 'legend.title_fontsize': 7,
}):
    g = sns.PairGrid(df_plot,
                     x_vars=x_vars,
                     y_vars=['*algorithm*'],
                     height=_height,
                     aspect=_panel_width / _height,
                     hue_order=order,
                     palette=sns.color_palette("rocket", n_colors=len(order)),
                    )

    g.map(sns.pointplot,
          orient="h",
          order=order,
          palette=sns.color_palette("rocket", n_colors=len(order)),
          join=False,
          estimator=np.mean,
          n_boot=1000,
          ci=95,
          markersize=4.3,
          linewidth=1.6,
          err_kws={'linewidth': 1.6},
         )

    titles = [pretty_labels.get(x, x.replace('_',' ').title().replace('(S)','(s)').replace('R2','$R^2$')) for x in x_vars]
    g.axes.flat[0].set_ylabel('')

    for ax, title in zip(g.axes.flat, titles):
        ax.set(title=title)
        ax.set_xlabel('')

        if any([n in title.lower() for n in ['size','time']]):
            ax.set_xscale('log')

        if title == '$R^2$ Test':
            ax.set_xlim([-.25,1])

        ax.yaxis.grid(True)

    # bold RASR entries; small square after neural methods
    g.fig.tight_layout(w_pad=0.4)
    g.fig.canvas.draw()
    _renderer = g.fig.canvas.get_renderer()
    _ax0 = g.axes.flat[0]
    for _label in _ax0.get_yticklabels():
        _name = _label.get_text()
        if is_rasr(_name):
            _label.set_fontweight('bold')
        if is_nsr(_name):
            _bbox = _label.get_window_extent(_renderer)
            _fx, _fy = g.fig.transFigure.inverted().transform(
                (_bbox.x1 + 2, (_bbox.y0 + _bbox.y1) / 2))
            g.fig.text(_fx, _fy, NSR_MARKER, color=NSR_MARKER_COLOR, fontsize=7,
                        va='center', ha='left', transform=g.fig.transFigure)

    sns.despine(left=True, bottom=True)

    # inlined save(): its tight_layout() would undo w_pad=0.4
    _name = "r2_test_rank_overview_plot_selected"
    print('saving', figdir+'/'+_name+'.pdf')
    if not os.path.exists(figdir):
        os.makedirs(figdir)
    plt.savefig(figdir+'/'+_name+'.pdf', bbox_inches='tight')